# Payments Lifecycle

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LineageLogic/LakeLogic/blob/main/examples/06_advanced_workflows/payments_lifecycle/payments_lifecycle.ipynb) 
[![GitHub Repo](https://img.shields.io/badge/GitHub-Repo-blue?logo=github)](https://github.com/LineageLogic/LakeLogic/blob/main/examples/06_advanced_workflows/payments_lifecycle/payments_lifecycle.ipynb)

## Business Scenario

Payments and customers evolve across Bronze, Silver, and Gold. You need a full lifecycle workflow that handles SCD2 and fact enrichment.

## Value Proposition

- End-to-end Bronze -> Silver -> Gold workflow
- SCD2 support for customer history
- Consistent fact table enrichment

---

## Goals

1. Load customer snapshots
2. Process payments
3. Build Gold facts


In [ ]:
import os
import pandas as pd
from lakelogic.core.processor import DataProcessor

# Ensure we use DuckDB
os.environ["LAKELOGIC_ENGINE"] = "duckdb"

## 2. Initialize Customers (Initial Load)
We process the first version of our customer data.

In [ ]:
# 1. Bronze
cust_bronze = DataProcessor(contract="01_bronze_customers.yaml")
res = cust_bronze.run_source()
cust_bronze.materialize(res.good)

# 2. Silver
cust_silver = DataProcessor(contract="02_silver_customers.yaml")
res = cust_silver.run_source()
cust_silver.materialize(res.good)

# 3. Gold (Creates SCD2 table)
cust_gold = DataProcessor(contract="03_gold_customers.yaml")
res = cust_gold.run_source()
cust_gold.materialize(res.good)

print("Initial Customer Dimension (Gold):")
print(pd.read_parquet("data/gold_customers_scd2.parquet")[['customer_id', 'name', 'plan', 'is_current']])

## 3. Process Payments (Joined with Customers)
Now we process payments, enriched with the initial customer metadata.

In [ ]:
# 1. Bronze
pay_bronze = DataProcessor(contract="01_bronze_payments.yaml")
res = pay_bronze.run_source()
pay_bronze.materialize(res.good)

# 2. Silver
pay_silver = DataProcessor(contract="02_silver_payments.yaml")
res = pay_silver.run_source()
pay_silver.materialize(res.good)

# 3. Gold (Enriched via Join)
pay_gold = DataProcessor(contract="03_gold_payments.yaml")
res = pay_gold.run_source()
pay_gold.materialize(res.good)

print("Enriched Payments (Gold):")
print(res.good[['raw_id', 'customer_id', 'amount_usd', 'customer_name', 'customer_plan']])

## 4. SCD Type 2: Customer Upgrade
Alice upgrades from 'pro' to 'enterprise'. We process the new version of customer data.
LakeLogic's `scd2` strategy will close the old record and open a new one.

In [ ]:
# Process v2 customers by overriding the source path
res = cust_bronze.run_source("data/raw_customers_v2.csv")
cust_bronze.materialize(res.good)

res = cust_silver.run_source()
cust_silver.materialize(res.good)

res = cust_gold.run_source()
cust_gold.materialize(res.good)

print("Updated Customer Dimension (Gold SCD2):")
df_cust = pd.read_parquet("data/gold_customers_scd2.parquet")
print(df_cust.sort_values(['customer_id', 'valid_from'])[['customer_id', 'name', 'plan', 'is_current', 'valid_from', 'valid_to']])

## 5. Re-Run Payments (Picking up current dimension)
Since the business logic points to the 'current' dimension record, running the payments again will now show Alice as 'enterprise'.

In [ ]:
res = pay_gold.run_source()
print("Payments with UPDATED Customer Metadata:")
print(res.good[['raw_id', 'customer_id', 'amount_usd', 'customer_name', 'customer_plan']])